# Zero-Positive Query Relabeling Pipeline

Recovers training signal from queries that have no positive labels (all 0s).

## Step 1: Install Dependencies

In [ ]:
!pip install faiss-cpu

## Step 2: Retrieve Candidates for Zero-Positive Queries

Uses E5 + TF-IDF (word/char) with RRF fusion, then BGE reranker to get top-30 candidates per query.

In [3]:
# One-Cell Pipeline (compact): E5 + RRF(E5 + word/char TF-IDF) + BGE reranker + cache + timing
import os, json, time, math, random, sys, subprocess, hashlib, re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer

# -------------------- config / env --------------------
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try: torch.set_float32_matmul_precision("high")
except: pass
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

CORPUS_JSONL = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")
TRAIN_JSONL  = os.getenv("TRAIN_JSONL",  "/content/hsrc_train.jsonl")

# === NEW/CHANGED ===
# We'll ignore NUM_QUERIES_TO_RUN so we don't truncate; keep env for compatibility but not used.
NUM_QUERIES_TO_RUN = os.getenv("NUM_QUERIES_TO_RUN", None)
NUM_QUERIES_TO_RUN = int(NUM_QUERIES_TO_RUN) if (NUM_QUERIES_TO_RUN and NUM_QUERIES_TO_RUN.isdigit()) else None

SHUFFLE_QUERIES    = bool(int(os.getenv("SHUFFLE_QUERIES", "0")))
K_TOTAL  = int(os.getenv("K_TOTAL",  "50"))
K_STAGE1 = int(os.getenv("K_STAGE1", "50"))
K_FINAL  = int(os.getenv("K_FINAL",  "20"))

# === NEW/CHANGED ===
# How many reranked results to SAVE for manual labeling (top-K after reranker)
SAVE_TOP_K = int(os.getenv("SAVE_TOP_K", "30"))
OUTPUT_JSONL = os.getenv("OUTPUT_JSONL", "/content/zero_pos_top30.jsonl")

# ---------- Lexical settings ----------
USE_TFIDF_LEXICAL     = bool(int(os.getenv("USE_TFIDF_LEXICAL", "1")))
ENABLE_CHAR_TFIDF     = bool(int(os.getenv("ENABLE_CHAR_TFIDF", "1")))
TFIDF_MAX_FEATS       = int(os.getenv("TFIDF_MAX_FEATS", "300000"))
TFIDF_WORD_NGRAM_MAX  = int(os.getenv("TFIDF_WORD_NGRAM_MAX", "2"))
TFIDF_CHAR_MIN        = int(os.getenv("TFIDF_CHAR_MIN", "3"))
TFIDF_CHAR_MAX        = int(os.getenv("TFIDF_CHAR_MAX", "5"))
TFIDF_CHAR_MAX_FEATS  = int(os.getenv("TFIDF_CHAR_MAX_FEATS", "300000"))

# Char-TFIDF subset scoring
CHAR_SUBSET           = bool(int(os.getenv("CHAR_SUBSET", "1")))
CHAR_SUBSET_MULT      = float(os.getenv("CHAR_SUBSET_MULT", "1.5"))
CHAR_SUBSET_MIN       = int(os.getenv("CHAR_SUBSET_MIN", "350"))

# ---------- RRF / weights ----------
RRF_K, RRF_POOL_MULT, RRF_POOL_MIN = int(os.getenv("RRF_K","60")), float(os.getenv("RRF_POOL_MULT","3")), int(os.getenv("RRF_POOL_MIN","150"))
SOURCE_WEIGHTS = {
    "e5":   float(os.getenv("WEIGHT_E5",   "0.55")),
    "lexw": float(os.getenv("WEIGHT_LEXW", "0.25")),
    "lexc": float(os.getenv("WEIGHT_LEXC", "0.20")),
}

E5_MODEL_NAME           = os.getenv("E5_MODEL_NAME", "intfloat/multilingual-e5-large")
BGE_RERANKER_MODEL_NAME = os.getenv("BGE_RERANKER_MODEL_NAME", "BAAI/bge-reranker-v2-m3")
CACHE_DIR, USE_MEMMAP_ON_LOAD = os.getenv("CACHE_DIR", "/content/embedding_cache_faiss"), bool(int(os.getenv("USE_MEMMAP_ON_LOAD","1")))
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] Lexical: {'TF-IDF(word+char)' if (USE_TFIDF_LEXICAL and ENABLE_CHAR_TFIDF) else ('TF-IDF(word)' if USE_TFIDF_LEXICAL else 'BM25(word)')}"
      f" | K_TOTAL={K_TOTAL} K_STAGE1={K_STAGE1} K_FINAL={K_FINAL} | SAVE_TOP_K={SAVE_TOP_K}")

# -------------------- deps (faiss/bm25) --------------------
try:
    import faiss; FAISS_AVAILABLE = True; print("FAISS: available.")
except Exception:
    pkg = "faiss-gpu" if torch.cuda.is_available() else "faiss-cpu"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    import faiss; FAISS_AVAILABLE = True; print("FAISS: installed.")
if not USE_TFIDF_LEXICAL:
    try:
        import rank_bm25; print("rank_bm25: available.")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"])
        import rank_bm25; print("rank_bm25: installed.")

# -------------------- tokenization --------------------
def _tok_bm25(s: str) -> List[str]:
    return (s or "").lower().split()

# -------------------- lexical wrappers --------------------
class GlobalLex:
    """
    mode: 'tfidf_word', 'tfidf_char', or 'bm25'
    """
    def __init__(self, texts: List[str], mode: str):
        self.mode = mode
        if mode == "tfidf_word":
            self.vec = TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, TFIDF_WORD_NGRAM_MAX),
                max_features=TFIDF_MAX_FEATS,
                token_pattern=r"(?u)[\u0590-\u05FFA-Za-z0-9_]{1,}"
            )
            self.mat = self.vec.fit_transform(texts)
        elif mode == "tfidf_char":
            self.vec = TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(TFIDF_CHAR_MIN, TFIDF_CHAR_MAX),
                max_features=TFIDF_CHAR_MAX_FEATS
            )
            self.mat = self.vec.fit_transform(texts)
        elif mode == "bm25":
            from rank_bm25 import BM25Okapi
            self.engine = BM25Okapi([_tok_bm25(t) for t in texts])
        else:
            raise ValueError(f"Unknown lexical mode: {mode}")

    def scores_all(self, q: str) -> np.ndarray:
        if self.mode.startswith("tfidf"):
            v = self.vec.transform([q])
            return (self.mat @ v.T).toarray().ravel().astype(np.float32)
        return np.asarray(self.engine.get_scores(_tok_bm25(q)), dtype=np.float32)

def _topk_from_scores(scores: np.ndarray, k:int)->List[Tuple[int,float]]:
    take = min(k, len(scores))
    part = np.argpartition(scores, -take)[-take:]
    ords = part[np.argsort(scores[part])[::-1]]
    return [(int(i), float(scores[i])) for i in ords]

def _topk_from_lex_engine(q: str, engine: Optional[GlobalLex], k: int) -> List[Tuple[int,float]]:
    if engine is None: return []
    return _topk_from_scores(engine.scores_all(q), k)

def _topk_from_lex_char_subset(q: str, engine: Optional[GlobalLex], subset: List[int], take_k:int) -> List[Tuple[int,float]]:
    if engine is None or not subset: return []
    v = engine.vec.transform([q])
    mat_sub = engine.mat[subset]
    scores = (mat_sub @ v.T).toarray().ravel().astype(np.float32)
    take = min(take_k, len(scores))
    part = np.argpartition(scores, -take)[-take:]
    ords = part[np.argsort(scores[part])[::-1]]
    return [(int(subset[i]), float(scores[i])) for i in ords]

# -------------------- E5 retriever --------------------
class E5Retriever:
    def __init__(self, name=None, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(name or E5_MODEL_NAME, use_fast=True)
        try:
            self.model = AutoModel.from_pretrained(name or E5_MODEL_NAME, torch_dtype=(torch.float16 if self.device=='cuda' else None), attn_implementation="sdpa").to(self.device)
        except TypeError:
            self.model = AutoModel.from_pretrained(name or E5_MODEL_NAME, torch_dtype=(torch.float16 if self.device=='cuda' else None)).to(self.device)
        self.model.eval()
        self.hidden_size = getattr(self.model.config, "hidden_size", 768)
    def _embed_batch(self, texts: List[str], max_len:int) -> torch.Tensor:
        enc = self.tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(self.device)
        with torch.inference_mode():
            out = self.model(**enc).last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1)
            emb = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
            return torch.nn.functional.normalize(emb, p=2, dim=1).cpu()
    def embed_texts(self, texts: List[str], is_query=False, batch_size=64) -> np.ndarray:
        pref = ("query: " if is_query else "passage: ")
        arrs = [self._embed_batch([pref+t.strip() for t in texts[i:i+batch_size]], 512 if not is_query else 512)
                for i in range(0, len(texts), batch_size)]
        return torch.cat(arrs, 0).numpy()

# -------------------- CE reranker --------------------
class BGEReranker:
    def __init__(self, name=None, device=None):
        from transformers import AutoModelForSequenceClassification
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(name or BGE_RERANKER_MODEL_NAME)
        try:
            self.model = AutoModelForSequenceClassification.from_pretrained(name or BGE_RERANKER_MODEL_NAME, torch_dtype=(torch.float16 if self.device=='cuda' else None), trust_remote_code=True, attn_implementation="sdpa").to(self.device)
        except TypeError:
            self.model = AutoModelForSequenceClassification.from_pretrained(name or BGE_RERANKER_MODEL_NAME, torch_dtype=(torch.float16 if self.device=='cuda' else None), trust_remote_code=True).to(self.device)
        self.model.eval()
    def rerank(self, q: str, passages: List[str], pids: List[str], top_k:int=20) -> List[Tuple[str,float]]:
        scores, bs = [], 4
        for i in range(0, len(passages), bs):
            b_pass, b_q = passages[i:i+bs], [q]*len(passages[i:i+bs])
            enc = self.tokenizer(b_q, b_pass, padding=True, truncation=True, max_length=512, return_tensors='pt').to(self.device)
            with torch.inference_mode():
                logits = self.model(**enc).logits
                if logits.ndim==1: s = logits
                elif logits.shape[1]==1: s = logits.squeeze(-1)
                else: s = logits[:,1]
            scores += s.detach().cpu().numpy().tolist()
        pairs = sorted(zip(pids, scores), key=lambda x:x[1], reverse=True)
        return pairs[:top_k]

# -------------------- utils: faiss / cache / rrf --------------------
def _build_faiss_index(embs: np.ndarray):
    if not FAISS_AVAILABLE: return None
    xb = np.asarray(embs, dtype=np.float32, order="C")
    index = faiss.IndexFlatIP(xb.shape[1])
    if torch.cuda.is_available() and hasattr(faiss, "StandardGpuResources"):
        try:
            res = faiss.StandardGpuResources(); index = faiss.index_cpu_to_gpu(res, 0, index)
            print("[FAISS] GPU IndexFlatIP.")
        except Exception as e: print("[FAISS] CPU index (fallback):", e)
    index.add(xb); print(f"[FAISS] {index.ntotal} vectors (dim={xb.shape[1]}).")
    return index

def _cache_key(path: str, n: int) -> str:
    p = Path(path); base = f"{p.name}|{p.stat().st_size if p.exists() else 0}|{int(p.stat().st_mtime) if p.exists() else 0}|{n}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()[:16]

def _cache_paths(model: str, key: str):
    tag = model.replace("/","__").replace(":","_").replace("@","_")
    base = f"{tag}__{key}"
    return (os.path.join(CACHE_DIR, base + ".npy"),
            os.path.join(CACHE_DIR, base + "_ids.json"),
            os.path.join(CACHE_DIR, base + ".meta.json"))

def load_cache(model:str, key:str, expect_n:int):
    e,i,m = _cache_paths(model,key)
    if not (os.path.exists(e) and os.path.exists(i) and os.path.exists(m)): return None
    try:
        meta = json.load(open(m,"r",encoding="utf-8"))
        if meta.get("model_name")!=model or meta.get("num_documents")!=expect_n: return None
        ids  = json.load(open(i,"r",encoding="utf-8"))
        embs = np.load(e, mmap_mode=("r" if USE_MEMMAP_ON_LOAD else None))
        if embs.shape[0]!=len(ids): return None
        return {"embeddings":embs, "ids":ids, "meta":meta}
    except Exception: return None

def save_cache(model:str, key:str, ids, embs: np.ndarray, meta_extra=None):
    e,i,m = _cache_paths(model,key)
    arr = np.asarray(embs, dtype=np.float32, order="C"); np.save(e, arr)
    json.dump(list(ids), open(i,"w",encoding="utf-8"), ensure_ascii=False)
    meta = {"model_name": model, "num_documents": len(ids), "dim": int(arr.shape[1])}
    if meta_extra: meta.update(meta_extra)
    json.dump(meta, open(m,"w",encoding="utf-8"))

def rrf_fuse_multi(sources: List[Tuple[str, List[Tuple[int,float]]]], k:int) -> List[int]:
    rank_maps = {}
    present = []
    for name, lst in sources:
        if lst:
            rank_maps[name] = {gi:r for r,(gi,_) in enumerate(lst)}
            present.append(name)
    if not present: return []
    wsum = sum(SOURCE_WEIGHTS.get(n, 0.0) for n in present) or 1.0
    weights = {n: SOURCE_WEIGHTS.get(n, 0.0) / wsum for n in present}
    big = 10**9
    all_ids = set().union(*[set(rank_maps[n].keys()) for n in present])
    fused = []
    for gi in all_ids:
        s = 0.0
        for n in present:
            r = rank_maps[n].get(gi, big)
            if r < big:
                s += weights[n] / (RRF_K + r)
        fused.append((gi, s))
    fused.sort(key=lambda x:x[1], reverse=True)
    return [gi for gi,_ in fused[:k]]

# -------------------- data io --------------------
def load_corpus(path:str):
    corpus={}
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            obj=json.loads(line); uid=obj.get("uuid") or obj.get("id")
            if uid is not None: corpus[uid]={"passage": obj.get("passage") or obj.get("text") or ""}
    return corpus

# === NEW/CHANGED === (capture query_uuid, and keep pos_ids logic)
def load_train(path:str, limit=None, shuffle=False):
    items=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o=json.loads(line)
            q=o.get("query","")
            qid=o.get("query_uuid") or o.get("id")  # capture query UUID if present
            gt={}; pos=set()
            paras=o.get("paragraphs",{}) or {}; labels=o.get("target_actions",{}) or {}
            for i in range(1000):
                p,fk=f"paragraph_{i}",f"target_action_{i}"
                if p not in paras or fk not in labels: break
                pid=paras[p].get("uuid") or paras[p].get("id")
                rel_raw=labels[fk]
                rel=int(rel_raw) if (isinstance(rel_raw,int) or (isinstance(rel_raw,str) and rel_raw.isdigit())) else int(float(rel_raw)) if rel_raw is not None else 0
                if pid is not None:
                    gt[pid]=rel
                    if rel>0: pos.add(pid)
            items.append({"query_uuid": qid, "query": q, "gt": gt, "pos_ids": pos})
    if shuffle: random.shuffle(items)
    if isinstance(limit,int) and limit>0:
        return items[:limit]
    return items  # default: all

# -------------------- preprocess (cache or compute) --------------------
def preprocess(corpus_dict: Dict[str, Dict[str,str]]):
    print("="*60, "\nPREPROCESS: E5 + BGE + Lexical(word [+ char])\n", "="*60)
    retriever, reranker = E5Retriever(E5_MODEL_NAME), BGEReranker(BGE_RERANKER_MODEL_NAME)
    corpus_ids = list(corpus_dict.keys())
    passages   = [corpus_dict[i].get('passage','') for i in corpus_ids]
    key        = _cache_key(CORPUS_JSONL, len(corpus_ids))
    cache      = load_cache(E5_MODEL_NAME, key, len(corpus_ids))
    if cache:
        print("Cache hit — restoring embeddings.")
        emb = cache["embeddings"]; corpus_ids = list(cache["ids"])
        passages = [corpus_dict[i]["passage"] for i in corpus_ids]
    else:
        print("Cache miss — computing E5 embeddings…")
        emb = retriever.embed_texts(passages, is_query=False, batch_size=128)
        save_cache(E5_MODEL_NAME, key, corpus_ids, emb, {"note":"E5 corpus embeddings cached (float32)"})
    faiss_idx  = _build_faiss_index(emb)

    if USE_TFIDF_LEXICAL:
        lex_word_engine = GlobalLex(passages, mode="tfidf_word")
        lex_char_engine = GlobalLex(passages, mode="tfidf_char") if ENABLE_CHAR_TFIDF else None
        lex_type = "tfidf(word" + ("+char" if ENABLE_CHAR_TFIDF else "") + ")"
    else:
        lex_word_engine = GlobalLex(passages, mode="bm25"); lex_char_engine=None
        lex_type = "bm25(word)"

    print("✓ Preprocess done.", f"E5 emb: {emb.shape} | LEX: {lex_type}")
    return {'retriever':retriever,'reranker':reranker,'corpus_ids':corpus_ids,'corpus_embeddings':emb,
            'corpus_texts':{doc:passages[i] for i,doc in enumerate(corpus_ids)},
            'faiss_index':faiss_idx,
            'lex_word_engine':lex_word_engine,'lex_char_engine':lex_char_engine,'lex_type':lex_type,
            'num_documents':len(corpus_ids)}

# -------------------- stage1 retrieval --------------------
def _topk_from_e5(qe: np.ndarray, pre:dict, k:int)->List[Tuple[int,float]]:
    if qe is None: return []
    idx = pre.get('faiss_index')
    if idx is not None:
        D,I = idx.search(qe, min(k, len(pre['corpus_ids'])))
        return [(int(i), float(d)) for d,i in zip(D[0], I[0]) if i>=0]
    sims = (qe @ pre['corpus_embeddings'].astype(np.float32).T)[0]
    ords = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in ords]

def stage1_ids(query_text:str, pre:dict, k:int)->List[str]:
    qe = pre['retriever'].embed_texts([query_text], is_query=True, batch_size=1).astype(np.float32) if SOURCE_WEIGHTS.get("e5",0)>0 else None
    pool_k = max(int(k*RRF_POOL_MULT), RRF_POOL_MIN)

    e5   = _topk_from_e5(qe, pre, pool_k)
    lexw = _topk_from_lex_engine(query_text, pre.get('lex_word_engine'), pool_k)

    # char TF-IDF scored only on subset (union of top-N from e5 & word)
    lexc = []
    if pre.get('lex_char_engine') is not None and CHAR_SUBSET:
        subset_size = max(int(pool_k * CHAR_SUBSET_MULT), CHAR_SUBSET_MIN)
        union_idx = sorted(set([gi for gi,_ in e5[:subset_size]] + [gi for gi,_ in lexw[:subset_size]]))
        lexc = _topk_from_lex_char_subset(query_text, pre['lex_char_engine'], union_idx, pool_k)
    elif pre.get('lex_char_engine') is not None:
        lexc = _topk_from_lex_engine(query_text, pre.get('lex_char_engine'), pool_k)

    fused_idx = rrf_fuse_multi([("e5", e5), ("lexw", lexw), ("lexc", lexc)], k)
    return [pre['corpus_ids'][gi] for gi in fused_idx]

# -------------------- predict --------------------
# === NEW/CHANGED === allow overriding top_k to SAVE_TOP_K
def predict(query: Dict[str,str], pre:dict, top_k:int=None):
    q = query.get('query','');  assert isinstance(q,str)
    tk = int(top_k) if top_k is not None else K_FINAL
    try:
        cand_ids = stage1_ids(q, pre, k=K_TOTAL)
        cand_pass = [pre['corpus_texts'].get(i,'') for i in cand_ids]
        pairs = pre['reranker'].rerank(q, cand_pass, cand_ids, top_k=tk)
        return [{'paragraph_uuid': pid, 'score': float(s)} for pid,s in pairs]
    except Exception as e:
        print("predict() error:", e)
        try:
            qe = pre['retriever'].embed_texts([q], is_query=True, batch_size=1)
            e5 = cosine_similarity(qe, pre['corpus_embeddings'])[0]
            top = np.argsort(e5)[::-1][:tk]
            return [{'paragraph_uuid': pre['corpus_ids'][i], 'score': float(e5[i])} for i in top]
        except Exception:
            return []

# -------------------- run: ONLY zero-positive queries, save JSONL --------------------
assert Path(CORPUS_JSONL).exists(), f"Corpus not found: {CORPUS_JSONL}"
assert Path(TRAIN_JSONL).exists(),  f"Train not found:  {TRAIN_JSONL}"
print("Loading corpus…"); corpus_dict = load_corpus(CORPUS_JSONL); print(f"Loaded {len(corpus_dict):,} docs.")
print("Loading queries…"); items = load_train(TRAIN_JSONL, limit=NUM_QUERIES_TO_RUN, shuffle=SHUFFLE_QUERIES)

total_queries = len(items)
num_with_pos = sum(1 for it in items if it["pos_ids"])
num_zero_pos = total_queries - num_with_pos
print(f"Total queries in train: {total_queries} | With ≥1 positive: {num_with_pos} | Zero-positive: {num_zero_pos}")

if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
print("\n=== Prepare (cache/embed) + FAISS + Lex ===")
t0=time.perf_counter(); pre = preprocess(corpus_dict)
if torch.cuda.is_available(): torch.cuda.synchronize()
prep_seconds = time.perf_counter()-t0
print(f"Preparation: {prep_seconds:.2f}s ({prep_seconds/60:.2f}m). FAISS: {'present' if pre.get('faiss_index') is not None else 'none'} | LEX: {pre['lex_type']}")

# === NEW/CHANGED === process only zero-positive queries and write JSONL
zero_pos_items = [it for it in items if not it["pos_ids"]]
print(f"\n=== Running pipeline on zero-positive queries only (n={len(zero_pos_items)}) ===")
out_path = Path(OUTPUT_JSONL)
with open(out_path, "w", encoding="utf-8") as wf:
    for i, it in enumerate(zero_pos_items, 1):
        qtext = it["query"]
        qid   = it.get("query_uuid")
        t=time.perf_counter()
        preds = predict({"query": qtext}, pre, top_k=SAVE_TOP_K)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed = time.perf_counter() - t
        if i<=5 or i%25==0:
            print(f"[{i:>4}/{len(zero_pos_items)}] {elapsed:.4f}s  | qid={qid}")

        # attach final rank (1-based) for manual labeling
        recs = []
        for rank, d in enumerate(preds, start=1):
            recs.append({
                "paragraph_uuid": d["paragraph_uuid"],
                "rank": rank,
                "score": d["score"],   # optional but useful
            })

        line = {
            "query_uuid": qid,
            "query": qtext,
            "top_candidates": recs
        }
        wf.write(json.dumps(line, ensure_ascii=False) + "\n")

print(f"\n✓ Done. Wrote top-{SAVE_TOP_K} reranked candidates for {len(zero_pos_items)} queries → {str(out_path)}")


[CONFIG] Lexical: TF-IDF(word+char) | K_TOTAL=50 K_STAGE1=50 K_FINAL=20 | SAVE_TOP_K=30
FAISS: available.
Loading corpus…
Loaded 127,731 docs.
Loading queries…
Total queries in train: 2034 | With ≥1 positive: 1929 | Zero-positive: 105
GPU: NVIDIA A100-SXM4-40GB

=== Prepare (cache/embed) + FAISS + Lex ===
PREPROCESS: E5 + BGE + Lexical(word [+ char])
Cache hit — restoring embeddings.
[FAISS] 127731 vectors (dim=1024).
✓ Preprocess done. E5 emb: (127731, 1024) | LEX: tfidf(word+char)
Preparation: 241.76s (4.03m). FAISS: present | LEX: tfidf(word+char)

=== Running pipeline on zero-positive queries only (n=105) ===
[   1/105] 0.4658s  | qid=d0c08ef6-2671-430c-86de-6b467e6127ae
[   2/105] 0.4735s  | qid=cc5136e9-e1db-4cae-b69b-b040d1a3c2dc
[   3/105] 0.4896s  | qid=2e23a41c-6707-4f5f-84bd-2f3c1d3259f6
[   4/105] 0.5002s  | qid=0ade51bf-a4ee-4480-87ce-0bd6ab484e78
[   5/105] 0.4836s  | qid=02e03c23-496c-42ad-aaa4-3ae56a8a4e67
[  25/105] 0.4908s  | qid=fffa2f6d-947b-43e4-a774-b752b543b80c
[

## Step 3: Preview Retrieved Candidates

Inspect the zero-positive queries and their top retrieved passages before LLM labeling.

In [ ]:
import os, json, textwrap
from pathlib import Path

ZERO_POS_JSONL = os.getenv("ZERO_POS_JSONL", "/content/zero_pos_top30.jsonl")
CORPUS_JSONL   = os.getenv("CORPUS_JSONL",   "/content/hsrc_corpus.jsonl")
N_TOP          = int(os.getenv("N_TOP", "10"))

wrap_width = int(os.getenv("WRAP_WIDTH", "100"))  # console wrap width for passages
preview_chars = int(os.getenv("PREVIEW_CHARS", "600"))  # shorten long passages in console

def load_corpus_map(path: str):
    """Load uuid -> passage map from corpus jsonl."""
    mp = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            o = json.loads(line)
            uid = o.get("uuid") or o.get("id")
            if uid:
                mp[uid] = (o.get("passage") or o.get("text") or "").strip()
    return mp

def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def shorten(txt: str, width_chars: int):
    # single-line + clip for previewing
    txt = (txt or "").replace("\n", " ").strip()
    if len(txt) <= width_chars:
        return txt
    return txt[:width_chars].rstrip() + "…"

def fmt_score(x):
    try:
        return f"{float(x):.4f}"
    except Exception:
        return str(x)

def main():
    assert Path(ZERO_POS_JSONL).exists(), f"Not found: {ZERO_POS_JSONL}"
    assert Path(CORPUS_JSONL).exists(),   f"Not found: {CORPUS_JSONL}"

    print(f"Loading corpus from: {CORPUS_JSONL}")
    corpus_map = load_corpus_map(CORPUS_JSONL)
    print(f"Loaded {len(corpus_map):,} passages.\n")

    items = list(iter_jsonl(ZERO_POS_JSONL))
    print(f"Loading zero-positive query results from: {ZERO_POS_JSONL}")
    print(f"Found {len(items):,} queries.\n")

    tw = textwrap.TextWrapper(width=wrap_width, subsequent_indent=" " * 6)

    for idx, rec in enumerate(items, 1):
        qid   = rec.get("query_uuid")
        query = (rec.get("query") or "").strip()
        cands = rec.get("top_candidates") or []

        print("=" * 80)
        print(f"Query {idx}/{len(items)}")
        print(f"query_uuid: {qid}")
        print(f"query: {query}\n")
        print(f"Top {min(N_TOP, len(cands))} candidates (final reranker order):")

        for j, c in enumerate(cands[:N_TOP], start=1):
            pid   = c.get("paragraph_uuid")
            rank  = c.get("rank", j)  # fall back to loop index if missing
            score = fmt_score(c.get("score"))
            passage = corpus_map.get(pid, "")
            preview = shorten(passage, preview_chars)

            header = f"  {rank:>2}. paragraph_uuid: {pid}  |  score: {score}"
            print(header)
            if preview:
                print("      " + tw.fill(preview))
            else:
                print("      (passage text not found in corpus)")
            print()  # blank line between candidates

    print("=" * 80)
    print("Done.")

if __name__ == "__main__":
    main()


## Step 4: Find Similar Positive Queries

For each zero-positive query, find similar queries that DO have positive labels using E5 embeddings. These serve as calibration anchors for LLM labeling.

In [ ]:
# Find top-N similar "positive" queries for each zero-positive query using E5 (symmetric: "query: " prefix)
import os, json, time
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# -------------------- Config (env) --------------------
ZERO_POS_JSONL = os.getenv("ZERO_POS_JSONL", "/content/zero_pos_top30.jsonl")
TRAIN_JSONL    = os.getenv("TRAIN_JSONL",    "/content/hsrc_train.jsonl")
E5_MODEL_NAME  = os.getenv("E5_MODEL_NAME",  "intfloat/multilingual-e5-large")
OUTPUT_JSONL   = os.getenv("OUTPUT_JSONL",   "/content/zero_pos_similar_queries_top3.jsonl")

TOP_N          = int(os.getenv("TOP_N", "3"))       # top-k similar positive queries per zero-pos query
BATCH_SIZE     = int(os.getenv("BATCH_SIZE", "64")) # E5 embed batch size
PRINT_FIRST    = int(os.getenv("PRINT_FIRST", "5")) # console preview count
MAX_LEN        = int(os.getenv("E5_MAX_LEN", "512"))

# -------------------- E5 Query Embedder --------------------
class E5QueryEmbedder:
    """
    Uses 'query: ' prefix for BOTH sides (symmetric semantic similarity),
    as recommended for E5 in similarity / clustering / paraphrase retrieval tasks.
    """
    def __init__(self, model_name: str = E5_MODEL_NAME):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        try:
            self.model = AutoModel.from_pretrained(
                model_name,
                torch_dtype=(torch.float16 if self.device == "cuda" else None),
                attn_implementation="sdpa"
            ).to(self.device)
        except TypeError:
            self.model = AutoModel.from_pretrained(
                model_name,
                torch_dtype=(torch.float16 if self.device == "cuda" else None)
            ).to(self.device)
        self.model.eval()

    @torch.inference_mode()
    def _embed_batch(self, texts: List[str]) -> torch.Tensor:
        enc = self.tokenizer(
            texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
        ).to(self.device)
        out = self.model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1)
        emb = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        return emb.detach().cpu()

    def embed_queries(self, queries: List[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
        # IMPORTANT: symmetric task → use "query: " prefix on BOTH sides
        prefixed = [("query: " + (q or "").strip()) for q in queries]
        chunks = []
        for i in range(0, len(prefixed), batch_size):
            chunks.append(self._embed_batch(prefixed[i:i+batch_size]))
        if not chunks:
            return np.zeros((0, 768), dtype=np.float32)
        return torch.cat(chunks, dim=0).numpy().astype(np.float32)

# -------------------- IO helpers --------------------
def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def load_zero_pos_queries(zero_pos_path: str) -> List[Dict]:
    """
    Expects JSONL lines of:
    { "query_uuid": ..., "query": ..., "top_candidates": [...] }
    """
    items = list(iter_jsonl(zero_pos_path))
    # Keep only query_uuid + query text
    return [{"query_uuid": it.get("query_uuid"), "query": it.get("query", "").strip()} for it in items]

def load_positive_queries(train_path: str) -> List[Dict]:
    """
    Reads the training jsonl, returns queries that have >=1 positive paragraph (label > 0).
    Output: list of dicts with {"query_uuid", "query"}.
    """
    pos_queries = []
    with open(train_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            o = json.loads(line)
            qid = o.get("query_uuid") or o.get("id")
            qtext = (o.get("query") or "").strip()
            paras = o.get("paragraphs", {}) or {}
            labels = o.get("target_actions", {}) or {}

            # detect any positive (>0)
            has_pos = False
            for i in range(1000):
                pk, lk = f"paragraph_{i}", f"target_action_{i}"
                if pk not in paras or lk not in labels:
                    break
                rel_raw = labels[lk]
                try:
                    rel = int(rel_raw) if (isinstance(rel_raw, int) or (isinstance(rel_raw, str) and rel_raw.isdigit())) else int(float(rel_raw))
                except Exception:
                    rel = 0
                if rel > 0:
                    has_pos = True
                    break
            if has_pos:
                pos_queries.append({"query_uuid": qid, "query": qtext})
    return pos_queries

# -------------------- Similarity search (cosine on normalized E5) --------------------
def topk_similar(zero_emb: np.ndarray, pos_emb: np.ndarray, k: int) -> List[List[int]]:
    """
    zero_emb: [Z, d], pos_emb: [P, d] — both L2-normalized.
    Returns indices (into pos_emb) of top-k for each zero query.
    """
    # cosine = dot product since embeddings are normalized
    sims = zero_emb @ pos_emb.T  # [Z, P]
    # argpartition for top-k per row
    Z, P = sims.shape
    k = min(k, P)
    # Get top-k indices sorted by score desc
    topk_idx = np.argpartition(sims, -k, axis=1)[:, -k:]
    # Now sort those k by similarity
    rows = []
    for i in range(Z):
        row = topk_idx[i]
        row_sorted = row[np.argsort(sims[i, row])[::-1]]
        rows.append(row_sorted.tolist())
    return rows

# -------------------- Main --------------------
def main():
    assert Path(ZERO_POS_JSONL).exists(), f"Not found: {ZERO_POS_JSONL}"
    assert Path(TRAIN_JSONL).exists(),    f"Not found: {TRAIN_JSONL}"

    zero_items = load_zero_pos_queries(ZERO_POS_JSONL)
    pos_items  = load_positive_queries(TRAIN_JSONL)

    print(f"Zero-positive queries: {len(zero_items)}")
    print(f"Positive-labeled queries: {len(pos_items)}")

    zero_q = [it["query"] for it in zero_items]
    pos_q  = [it["query"] for it in pos_items]

    # Embed (symmetric → 'query: ' on BOTH)
    e5 = E5QueryEmbedder(E5_MODEL_NAME)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    t0 = time.perf_counter()
    zero_emb = e5.embed_queries(zero_q, batch_size=BATCH_SIZE)
    pos_emb  = e5.embed_queries(pos_q,  batch_size=BATCH_SIZE)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    print(f"Embedded {len(zero_q)} zero-pos and {len(pos_q)} positive queries in {elapsed:.2f}s.")

    # Find top-N positive neighbors for each zero-pos query
    top_idx_per_zero = topk_similar(zero_emb, pos_emb, k=TOP_N)
    print("Computing similarities and writing output…")

    out_path = Path(OUTPUT_JSONL)
    with open(out_path, "w", encoding="utf-8") as wf:
        for zi, zero_it in enumerate(zero_items):
            neighbors = []
            for rank, pi in enumerate(top_idx_per_zero[zi], start=1):
                neighbors.append({
                    "similar_query_uuid": pos_items[pi]["query_uuid"],
                    "similar_query": pos_items[pi]["query"],
                    "rank": rank,
                    "similarity": float(np.dot(zero_emb[zi], pos_emb[pi])),  # cosine
                })
            line = {
                "query_uuid": zero_it["query_uuid"],
                "query": zero_it["query"],
                "top_similar": neighbors
            }
            wf.write(json.dumps(line, ensure_ascii=False) + "\n")

    print(f"✓ Done. Wrote results to: {str(out_path)}")

    # Console preview
    show = min(PRINT_FIRST, len(zero_items))
    if show > 0:
        print("\nPreview:")
        for i, zero_it in enumerate(zero_items[:show], 1):
            print("=" * 80)
            print(f"[{i}/{len(zero_items)}] query_uuid: {zero_it['query_uuid']}")
            print(f"query: {zero_it['query']}")
            print("Top similar positives:")
            for nb in json.loads(next(open(out_path, "r", encoding="utf-8")) if i==1 else "{}") if False else []:
                pass  # (no-op; avoid re-reading file per item)
        # Instead, re-read only the first `show` lines cleanly:
        with open(out_path, "r", encoding="utf-8") as f:
            for i in range(show):
                rec = json.loads(next(f))
                print("=" * 80)
                print(f"[{i+1}/{len(zero_items)}] query_uuid: {rec['query_uuid']}")
                print(f"query: {rec['query']}")
                for nb in rec["top_similar"]:
                    print(f"  {nb['rank']}. ({nb['similarity']:.4f})  qid={nb['similar_query_uuid']}")
                    print(f"     {nb['similar_query']}")
        print("=" * 80)

if __name__ == "__main__":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    main()


## Step 5: Build LLM Input Package

Creates JSONL with: (A) label-0 candidates, (B) top-10 retrieved passages, (C) similar positive queries with labeled examples for calibration.

In [6]:
# Build a JSONL for LLM relabeling of zero-positive queries
# Includes: (A) label-0 training candidates, (B) top-10 pipeline picks, (C) 2 similar positive queries with ~10 diverse labeled examples
import os, json, sys, collections, textwrap
from pathlib import Path
from typing import Dict, List, Tuple, Any

# ---------------- Env / Defaults ----------------
TRAIN_JSONL   = os.getenv("TRAIN_JSONL",   "/content/hsrc_train.jsonl")
CORPUS_JSONL  = os.getenv("CORPUS_JSONL",  "/content/hsrc_corpus.jsonl")
ZERO_POS_JSONL= os.getenv("ZERO_POS_JSONL","/content/zero_pos_top30.jsonl")  # from your earlier pipeline
SIMILAR_JSONL = os.getenv("SIMILAR_JSONL", "/content/zero_pos_similar_queries_top3.jsonl")  # from the E5 similarity script
OUTPUT_JSONL  = os.getenv("OUTPUT_JSONL",  "/content/llm_relabel_package.jsonl")

TOP_PIPELINE_N   = int(os.getenv("TOP_PIPELINE_N", "10"))   # how many pipeline picks to include
DIVERSE_PER_POSQ = int(os.getenv("DIVERSE_PER_POSQ", "10")) # how many labeled passages to include for each similar positive query
NUM_SIM_QUERIES  = int(os.getenv("NUM_SIM_QUERIES", "2"))   # take top-2 most similar positives
WRITE_INSTRUCTIONS = bool(int(os.getenv("WRITE_INSTRUCTIONS", "1")))
INSTR_PATH = os.getenv("INSTR_PATH", "/content/instructions.md")

# ---------------- Helpers ----------------
def load_corpus_map(path: str) -> Dict[str, str]:
    mp = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            uid = o.get("uuid") or o.get("id")
            if uid:
                mp[uid] = (o.get("passage") or o.get("text") or "").strip()
    return mp

def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def load_zero_pos(zero_pos_path: str) -> Dict[str, Dict[str, Any]]:
    """
    Returns map: query_uuid -> {query, top_candidates:[{paragraph_uuid, rank, score}, ...]}
    (Assumes zero_pos_top30.jsonl format produced earlier)
    """
    mp = {}
    for o in iter_jsonl(zero_pos_path):
        qid = o.get("query_uuid")
        if qid:
            mp[qid] = {
                "query": o.get("query", "").strip(),
                "top_candidates": o.get("top_candidates") or []
            }
    return mp

def load_similar(similar_path: str) -> Dict[str, List[Dict[str, Any]]]:
    """
    Returns map: zero_qid -> list of {similar_query_uuid, similar_query, rank, similarity}
    """
    mp = {}
    for o in iter_jsonl(similar_path):
        qid = o.get("query_uuid")
        if not qid: continue
        sim = o.get("top_similar") or []
        mp[qid] = sim
    return mp

def load_train_queries(train_path: str):
    """
    Returns:
      train_by_qid: { qid: { "query": str, "labeled": [ {paragraph_uuid, passage, label, idx} *up to 20 ] } }
      zero_qids: set of qids with all labels == 0
      pos_qids:  set of qids with any label > 0
    """
    train_by_qid = {}
    zero_qids, pos_qids = set(), set()
    with open(train_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            qid   = o.get("query_uuid") or o.get("id")
            query = (o.get("query") or "").strip()
            paras = o.get("paragraphs", {}) or {}
            labels= o.get("target_actions", {}) or {}

            labeled = []
            has_pos = False
            for i in range(1000):
                pk, lk = f"paragraph_{i}", f"target_action_{i}"
                if pk not in paras or lk not in labels: break
                pid = paras[pk].get("uuid") or paras[pk].get("id")
                passage = (paras[pk].get("passage") or paras[pk].get("text") or "").strip()
                rel_raw = labels[lk]
                try:
                    label = int(rel_raw) if (isinstance(rel_raw,int) or (isinstance(rel_raw,str) and rel_raw.isdigit())) else int(float(rel_raw))
                except Exception:
                    label = 0
                labeled.append({"paragraph_uuid": pid, "passage": passage, "label": label, "candidate_index": i})
                if label > 0: has_pos = True

            if qid:
                train_by_qid[qid] = {"query": query, "labeled": labeled}
                (pos_qids if has_pos else zero_qids).add(qid)
    return train_by_qid, zero_qids, pos_qids

def select_diverse_labeled(labeled: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    """
    Pick up to k examples with diverse labels (aim for coverage of 4..0).
    Strategy:
      1) group by label
      2) round-robin over labels [4,3,2,1,0], preserving original order
    """
    by_label = collections.defaultdict(list)
    for ex in labeled:
        by_label[ex["label"]].append(ex)

    order = [4,3,2,1,0]
    ptr = {L:0 for L in order}
    out = []
    while len(out) < k:
        progressed = False
        for L in order:
            lst = by_label.get(L, [])
            j = ptr[L]
            if j < len(lst):
                out.append(lst[j])
                ptr[L] = j + 1
                progressed = True
                if len(out) >= k:
                    break
        if not progressed:
            break
    return out

def main():
    # ---------- existence checks ----------
    for p in [TRAIN_JSONL, CORPUS_JSONL, ZERO_POS_JSONL, SIMILAR_JSONL]:
        assert Path(p).exists(), f"Missing required file: {p}"

    # ---------- loads ----------
    print("Loading corpus map…")
    corpus_map = load_corpus_map(CORPUS_JSONL)

    print("Loading train queries…")
    train_by_qid, zero_qids_train, pos_qids_train = load_train_queries(TRAIN_JSONL)
    print(f"Train queries: {len(train_by_qid):,} | zero-pos in train: {len(zero_qids_train):,} | positive: {len(pos_qids_train):,}")

    print("Loading zero-pos pipeline results…")
    zero_pkg = load_zero_pos(ZERO_POS_JSONL)
    print(f"Zero-pos queries with pipeline results: {len(zero_pkg):,}")

    print("Loading similar-positive mapping…")
    sim_map = load_similar(SIMILAR_JSONL)

    # ---------- build JSONL ----------
    out_path = Path(OUTPUT_JSONL)
    written = 0
    with open(out_path, "w", encoding="utf-8") as wf:
        for qid, zp in zero_pkg.items():
            query_text = zp["query"]
            # (A) training labeled (all 0 for this qid)
            labeled_train = train_by_qid.get(qid, {}).get("labeled", [])
            labeled_zero = [ {"paragraph_uuid": ex["paragraph_uuid"], "passage": ex["passage"], "label": 0}
                             for ex in labeled_train if ex.get("label",0)==0 ]

            # (B) pipeline top-10 with passage text
            cands = []
            for d in (zp.get("top_candidates") or [])[:TOP_PIPELINE_N]:
                pid = d.get("paragraph_uuid")
                cands.append({
                    "paragraph_uuid": pid,
                    "passage": corpus_map.get(pid, ""),
                    "rank": int(d.get("rank", 0)) if isinstance(d.get("rank", 0), int) else int(d.get("rank", 0) or 0),
                    "score": float(d.get("score", 0.0))
                })

            # (C) two most similar positive queries, each with ~10 diverse labeled examples
            sim_entries = sim_map.get(qid, [])[:NUM_SIM_QUERIES]
            similar_queries = []
            for se in sim_entries:
                pos_qid = se.get("similar_query_uuid")
                if not pos_qid or pos_qid not in train_by_qid:
                    continue
                pos_info = train_by_qid[pos_qid]
                # diverse labeled selection from that pos query
                diverse = select_diverse_labeled(pos_info["labeled"], k=DIVERSE_PER_POSQ)
                diverse = [
                    {
                        "paragraph_uuid": ex["paragraph_uuid"],
                        "passage": ex["passage"],
                        "label": int(ex["label"])
                    } for ex in diverse
                ]
                similar_queries.append({
                    "similar_query_uuid": pos_qid,
                    "similar_query": pos_info["query"],
                    "similarity": float(se.get("similarity", 0.0)),
                    "diverse_labeled_examples": diverse
                })

            line = {
                "query_uuid": qid,
                "query": query_text,
                "labeled_negatives": labeled_zero,              # part A (all label-0)
                "pipeline_topk": cands,                         # part B (top-10)
                "supporting_positive_queries": similar_queries  # part C (two neighbors with diverse labels)
            }
            wf.write(json.dumps(line, ensure_ascii=False) + "\n")
            written += 1

    print(f"✓ Wrote {written} zero-positive query packages → {str(out_path)}")

    # ---------- optional: write instructions ----------

if __name__ == "__main__":

  main()

Loading corpus map…
Loading train queries…
Train queries: 2,034 | zero-pos in train: 105 | positive: 1,929
Loading zero-pos pipeline results…
Zero-pos queries with pipeline results: 105
Loading similar-positive mapping…
✓ Wrote 105 zero-positive query packages → /content/llm_relabel_package.jsonl


## Step 6: LLM Relabeling (Pass 1)

Calls GPT-5 via OpenAI API to assign 0-4 labels to the top-10 retrieved passages.

In [ ]:
# relabel_from_llm_package.py
# Uses your existing llm_relabel_package.jsonl to ask an LLM for labels (0..4)
# for the top-10 reranked passages of each non-positive query.
#
# Inputs:
#   PACKAGE_JSONL (default: /content/llm_relabel_package.jsonl)
#   CORPUS_JSONL  (default: /content/hsrc_corpus.jsonl)  # fallback text lookup if needed
#
# Output:
#   OUT_JSONL     (default: ./zero_pos_pseudo_labels.jsonl)
#
# Env knobs:
#   TOP_PIPELINE_N=10 MODEL=gpt-5 CONCURRENCY=10 MAX_RETRY=2
#   REQUIRE_CALIBRATION=0  (set to 1 to skip items without calibration examples)
#   PRINT_FIRST=3          (preview rows after completion)
#   DEBUG=1                (set 0 to reduce logs)
#
# Author: you, with ❤️

import os, json, asyncio, sys
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional
from openai import AsyncOpenAI

# ---------------- API KEY (set here OR via env var OPENAI_API_KEY) ----------------
# Fill in your key here, or leave "" to use the OPENAI_API_KEY env var.
OPENAI_API_KEY = "your-key-here"  # e.g., "sk-..."  (leave empty to use environment)

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-key-here")

if not OPENAI_API_KEY:
    raise RuntimeError("OpenAI API key missing. Set OPENAI_API_KEY at the top or export OPENAI_API_KEY env var.")

# ---------------- Config (env) ----------------
PACKAGE_JSONL = os.getenv("PACKAGE_JSONL", "./llm_relabel_package.jsonl")
CORPUS_JSONL  = os.getenv("CORPUS_JSONL",  "./hsrc_corpus.jsonl")  # optional fallback lookup
OUT_JSONL     = os.getenv("OUT_JSONL",     "./zero_pos_pseudo_labels.jsonl")

TOP_PIPELINE_N       = int(os.getenv("TOP_PIPELINE_N", "8"))  # how many unlabeled to send (cap)
MODEL                = os.getenv("MODEL", "gpt-5")
CONCURRENCY          = int(os.getenv("CONCURRENCY", "5"))
MAX_RETRY            = int(os.getenv("MAX_RETRY", "2"))
PRINT_FIRST          = int(os.getenv("PRINT_FIRST", "3"))  # console preview
REQUIRE_CALIBRATION  = bool(int(os.getenv("REQUIRE_CALIBRATION", "0")))
DEBUG                = bool(int(os.getenv("DEBUG", "1")))

def _dbg(msg: str):
    if DEBUG:
        print(msg, flush=True)

print("[CFG] Starting relabel_from_llm_package.py")
print(f"[CFG] PACKAGE_JSONL={PACKAGE_JSONL}")
print(f"[CFG] CORPUS_JSONL={CORPUS_JSONL}")
print(f"[CFG] OUT_JSONL={OUT_JSONL}")
print(f"[CFG] TOP_PIPELINE_N={TOP_PIPELINE_N} MODEL={MODEL} CONCURRENCY={CONCURRENCY} MAX_RETRY={MAX_RETRY}")
print(f"[CFG] REQUIRE_CALIBRATION={REQUIRE_CALIBRATION} DEBUG={DEBUG}")

# ---------------- Prompt (CALIBRATION FIRST) ----------------
DEV_PROMPT = (
    "You are a Hebrew relevance judge.\n\n"
    "INPUT FORMAT (one JSON object):\n"
    "{\n"
    "  \"calibration\": [                 // read FIRST: similar queries w/ labeled examples\n"
    "    {\n"
    "      \"similar_query\": \"<Hebrew query>\",\n"
    "      \"examples\": [               // labeled examples for that similar query\n"
    "        {\"label\": 4, \"passage\": \"...\"},\n"
    "        {\"label\": 0, \"passage\": \"...\"}\n"
    "      ]\n"
    "    },\n"
    "    { \"similar_query\": \"...\", \"examples\": [ ... ] }\n"
    "  ],\n"
    "  \"target\": {                      // then label THIS target query\n"
    "    \"query\": \"<Hebrew query>\",\n"
    "    \"to_label\": [                 // UNLABELED passages (top-10 reranked)\n"
    "      {\"passage\": \"...\"},\n"
    "      {\"passage\": \"...\"}\n"
    "    ]\n"
    "  }\n"
    "}\n\n"
    "LABEL SCALE (return an integer):\n"
    "0 — לא קשור (אין מענה לשאילתה)\n"
    "1 — קשר חלש (אזכור נושא/מונחים בלי מידע שימושי לשאילתה)\n"
    "2 — חלקי (מענה חלקי/משמעותי אך חסרים רכיבים מרכזיים)\n"
    "3 — חזק (מכסה כמעט הכול; אולי חסר פרט קטן)\n"
    "4 — מלא (מענה ברור ושלם לשאילתה)\n\n"
    "TASK:\n"
    "1) Study the CALIBRATION blocks first to internalize how labels 0–4 look for similar intents.\n"
    "2) For each item in target.to_label, assign exactly one label in {0,1,2,3,4} relative to target.query.\n\n"
    "OUTPUT (STRICT JSON, no markdown/comments, no extra keys):\n"
    "{\n"
    "  \"labels\": [ <0|1|2|3|4>, <0|1|2|3|4>, ... ]\n"
    "}\n\n"
    "CONSTRAINTS:\n"
    "- Preserve order: i-th output label corresponds to i-th passage in target.to_label.\n"
    "- Output ONLY valid JSON, no extra keys.\n"
)

# ---------------- IO helpers ----------------
def read_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                yield json.loads(s)

def write_jsonl(path: str, rows: Iterable[Dict[str, Any]]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def parse_json_loose(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except Exception:
        pass
    t = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    a, b = t.find("{"), t.rfind("}")
    if a >= 0 and b > a:
        return json.loads(t[a:b+1])
    raise ValueError("Could not parse JSON from model output")

# Optional: lazy corpus lookup if any passage is missing in the package
_corpus_map: Optional[Dict[str, str]] = None
def _get_corpus_map() -> Dict[str, str]:
    global _corpus_map
    if _corpus_map is not None:
        return _corpus_map
    _corpus_map = {}
    if Path(CORPUS_JSONL).exists():
        _dbg(f"[IO] Loading corpus map from {CORPUS_JSONL} …")
        count = 0
        for o in read_jsonl(CORPUS_JSONL):
            uid = o.get("uuid") or o.get("id")
            if uid:
                _corpus_map[uid] = (o.get("passage") or o.get("text") or "").strip()
                count += 1
                if DEBUG and count % 20000 == 0:
                    _dbg(f"[IO]  … loaded {count:,} passages")
        _dbg(f"[IO] Corpus map loaded: {len(_corpus_map):,} entries")
    else:
        _dbg(f"[IO] Corpus file not found ({CORPUS_JSONL}); skipping fallback map.")
    return _corpus_map

# ---------------- Payload builder (CALIBRATION before TARGET) ----------------
def make_llm_payload(pkg: Dict[str, Any], top_n: int = TOP_PIPELINE_N) -> Dict[str, Any]:
    """
    pkg fields (from llm_relabel_package.jsonl):
      - query_uuid, query
      - pipeline_topk: [{paragraph_uuid, passage, rank, score}, ...]
      - supporting_positive_queries: [
            {
              "similar_query": "...",
              "diverse_labeled_examples": [{"passage","label"}, ...]
            }, ...
        ]
    Returns payload:
      {
        "calibration": [
          {"similar_query": "...", "examples": [{"label":X,"passage":"..."} ...]},
          ...
        ],
        "target": {
          "query": "...",
          "to_label": [{"passage":"..."}, ...]
        }
      }
    """
    # --- calibration: keep each similar query grouped with its labeled examples ---
    calibration = []
    sim_list = pkg.get("supporting_positive_queries") or []
    _dbg(f"[DBG] Building calibration for query_uuid={pkg.get('query_uuid')} with {len(sim_list)} similar queries")
    for idx, sim in enumerate(sim_list, start=1):
        sq = (sim.get("similar_query") or "").strip()
        exs = []
        for ex in (sim.get("diverse_labeled_examples") or []):
            exs.append({
                "label": int(ex.get("label", 0)),
                "passage": (ex.get("passage") or "").strip(),
            })
        _dbg(f"[DBG]  - similar[{idx}] examples={len(exs)}")
        if sq or exs:
            calibration.append({"similar_query": sq, "examples": exs})

    # --- target: ensure we have text for each passage (fallback to corpus if needed) ---
    qtext = (pkg.get("query") or "").strip()
    to_label_src = (pkg.get("pipeline_topk") or [])[:top_n]
    to_label: List[Dict[str, Any]] = []
    missing = 0
    for t in to_label_src:
        pid = t.get("paragraph_uuid")
        passage = (t.get("passage") or "").strip()
        if not passage and pid:
            missing += 1
            passage = _get_corpus_map().get(pid, "").strip()
        to_label.append({"passage": passage})
    _dbg(f"[DBG] Target build: qid={pkg.get('query_uuid')} to_label={len(to_label)} (filled_missing_from_corpus={missing})")

    return {"calibration": calibration, "target": {"query": qtext, "to_label": to_label}}

# ---------------- Async LLM calling ----------------
async def call_model(client: AsyncOpenAI, model: str, payload: Dict[str, Any]) -> List[int]:
    backoff = 1.0
    last_err = None
    for attempt in range(MAX_RETRY):
        try:
            _dbg(f"[CALL] Model attempt {attempt+1} | cal_blocks={len(payload.get('calibration', []))} | to_label={len(payload.get('target', {}).get('to_label', []))}")
            resp = await client.responses.create(
                model=model,
                input=[
                    {"role": "developer", "content": [{"type": "input_text", "text": DEV_PROMPT}]},
                    {"role": "user", "content": [{"type": "input_text", "text": json.dumps(payload, ensure_ascii=False)}]}
                ],
                text={"format": {"type": "text"}, "verbosity": "medium"},
                reasoning={"effort": "medium", "summary": "auto"},
                tools=[],
                store=False
            )
            # Extract text
            text = getattr(resp, "output_text", None)
            if not text and hasattr(resp, "output") and isinstance(resp.output, list):
                parts = []
                for item in resp.output:
                    for c in getattr(item, "content", []) or []:
                        txt = getattr(c, "text", None)
                        if isinstance(txt, str):
                            parts.append(txt)
                if parts:
                    text = "\n".join(parts)
            if not text and hasattr(resp, "content") and isinstance(resp.content, list):
                parts = []
                for c in resp.content:
                    txt = getattr(c, "text", None)
                    if isinstance(txt, str):
                        parts.append(txt)
                if parts:
                    text = "\n".join(parts)
            if not text:
                raise RuntimeError("Empty response")

            _dbg(f"[CALL] Raw output (first 400 chars): {text[:400].replace(os.linesep, ' ')}{'…' if len(text) > 400 else ''}")
            out = parse_json_loose(text)
            labels = out.get("labels")
            if not isinstance(labels, list) or not all(isinstance(x, int) for x in labels):
                raise ValueError("Invalid labels format")
            for x in labels:
                if x < 0 or x > 4:
                    raise ValueError(f"Label out of range: {x}")
            _dbg(f"[CALL] Parsed labels: {labels}")
            return labels
        except Exception as e:
            _dbg(f"[ERR] Model call error on attempt {attempt+1}: {e}")
            last_err = e
            if attempt == MAX_RETRY - 1:
                raise
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 8.0)
    raise last_err or RuntimeError("Unknown error")

# ---------------- Main pipeline ----------------
async def main():
    assert Path(PACKAGE_JSONL).exists(), f"Missing input: {PACKAGE_JSONL}"

    client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    Path(OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)

    # If OUT exists, resume by skipping already-labeled query_uuids
    done = set()
    if Path(OUT_JSONL).exists():
        _dbg(f"[IO] Resuming from existing OUT_JSONL: {OUT_JSONL}")
        for row in read_jsonl(OUT_JSONL):
            qid = row.get("query_uuid")
            if qid:
                done.add(qid)
        _dbg(f"[IO] Already labeled queries found: {len(done)}")

    sem = asyncio.Semaphore(CONCURRENCY)
    lock = asyncio.Lock()
    out_fh = open(OUT_JSONL, "a" if done else "w", encoding="utf-8")

    async def worker(pkg: Dict[str, Any]):
        qid = pkg.get("query_uuid")
        if not qid:
            _dbg("[DBG] Skipping a package with missing query_uuid")
            return
        if qid in done:
            _dbg(f"[DBG] Skipping already completed query_uuid={qid}")
            return

        payload = make_llm_payload(pkg, top_n=TOP_PIPELINE_N)
        cal_blocks = len(payload.get("calibration", []))
        to_label_n = len(payload.get("target", {}).get("to_label", []))
        _dbg(f"[DBG] Worker qid={qid} | calibration_blocks={cal_blocks} | to_label={to_label_n}")

        if not to_label_n:
            _dbg(f"[DBG] No items to label for qid={qid}; skipping.")
            return
        if REQUIRE_CALIBRATION and cal_blocks == 0:
            _dbg(f"[DBG] REQUIRE_CALIBRATION=1 and no calibration for qid={qid}; skipping.")
            return

        labels = await call_model(client, MODEL, payload)

        # Align back to paragraph UUIDs (preserve order of pipeline_topk we sent)
        topk = (pkg.get("pipeline_topk") or [])[:len(labels)]
        aligned_count = min(len(labels), len(topk))
        if aligned_count == 0:
            _dbg(f"[DBG] Alignment produced zero items for qid={qid}; skipping write.")
            return

        rows = []
        for lab, src in zip(labels[:aligned_count], topk[:aligned_count]):
            rows.append({
                "paragraph_uuid": src.get("paragraph_uuid"),
                "label": int(lab),
                "rank": src.get("rank"),
                "score": src.get("score")
            })

        out_row = {
            "query_uuid": qid,
            "query": pkg.get("query"),
            "labels": rows
        }
        async with lock:
            out_fh.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            out_fh.flush()
        _dbg(f"[IO] Wrote labels for qid={qid} (n={len(rows)})")

    tasks = []
    count = 0
    for pkg in read_jsonl(PACKAGE_JSONL):
        count += 1
        async def run(pkg=pkg):
            async with sem:
                try:
                    await worker(pkg)
                except Exception as e:
                    print(f"[ERR] {pkg.get('query_uuid','?')}: {e}", file=sys.stderr)
        tasks.append(asyncio.create_task(run()))

    print(f"[IO] Scheduling {count} packages…")
    await asyncio.gather(*tasks)
    out_fh.close()

    # Preview
    print(f"[DONE] wrote → {OUT_JSONL}")
    if PRINT_FIRST > 0 and Path(OUT_JSONL).exists():
        print("\n[PREVIEW]")
        for i, row in enumerate(read_jsonl(OUT_JSONL)):
            if i >= PRINT_FIRST: break
            s = json.dumps(row, ensure_ascii=False)
            print(s[:800] + ("…" if len(s) > 800 else ""))

if __name__ == "__main__":
    asyncio.run(main())


## Step 7: LLM Relabeling (Pass 2)

Second pass with stricter prompt and calibration from similar queries for more accurate labels.

In [ ]:
# relabel_pass2_from_pseudo.py
# Second pass: relabel ALL pseudo-labeled passages for the 105 queries in zero_pos_pseudo_labels.jsonl,
# using calibration (similar queries) and a stricter prompt.
#
# Inputs:
#   PSEUDO_JSONL=./zero_pos_pseudo_labels.jsonl
#   LLM_PACKAGE_JSONL=./llm_relabel_package.jsonl
#   CORPUS_JSONL=./hsrc_corpus.jsonl
# Output:
#   OUT_JSONL=./zero_pos_pseudo_labels.pass2.jsonl
#
# Knobs:
#   MODEL=gpt-5 CONCURRENCY=8 MAX_RETRY=2
#   MAX_SIMS=2 (0=all)   MAX_EXAMPLES_PER_SIM=5
#   PRINT_FIRST=3  DEBUG=1

import os, json, asyncio, sys, re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional
from openai import AsyncOpenAI

# ---------------- API KEY ----------------
OPENAI_API_KEY = "your-key-here"  # e.g., "sk-..."; leave "" to read from env
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-key-here")
if not OPENAI_API_KEY:
    raise RuntimeError("OpenAI API key missing. Set OPENAI_API_KEY here or as an env var.")

# ---------------- Config ----------------
PSEUDO_JSONL       = os.getenv("PSEUDO_JSONL",       "./zero_pos_pseudo_labels.jsonl")
LLM_PACKAGE_JSONL  = os.getenv("LLM_PACKAGE_JSONL",  "./llm_relabel_package.jsonl")
CORPUS_JSONL       = os.getenv("CORPUS_JSONL",       "./hsrc_corpus.jsonl")
OUT_JSONL          = os.getenv("OUT_JSONL",          "./zero_pos_pseudo_labels.pass2.jsonl")

MODEL              = os.getenv("MODEL", "gpt-5")
CONCURRENCY        = int(os.getenv("CONCURRENCY", "5"))
MAX_RETRY          = int(os.getenv("MAX_RETRY", "2"))
PRINT_FIRST        = int(os.getenv("PRINT_FIRST", "3"))
DEBUG              = bool(int(os.getenv("DEBUG", "1")))
MAX_SIMS           = int(os.getenv("MAX_SIMS", "1"))   # 0 = all similar queries
MAX_EXAMPLES_PER_SIM = int(os.getenv("MAX_EXAMPLES_PER_SIM", "6"))

def _dbg(msg: str):
    if DEBUG:
        print(msg, flush=True)

# ---------------- Prompt: calibration FIRST, stricter rules ----------------
DEV_PROMPT = (
    "You are a Hebrew relevance judge for passage retrieval.\n"
    "\n"
    "INPUT is one JSON object with two parts:\n"
    "{\n"
    "  \"calibration\": [\n"
    "    {\"similar_query\": \"...\", \"examples\": [\n"
    "       {\"label\": 4, \"passage\": \"...\"}, {\"label\": 2, \"passage\": \"...\"}\n"
    "    ]},\n"
    "    ...\n"
    "  ],\n"
    "  \"target\": {\n"
    "    \"query\": \"<Hebrew query>\",\n"
    "    \"to_label\": [\n"
    "      {\"passage\": \"...\", \"pid\": \"<uuid>\", \"plabel\": <0..4>},\n"
    "      ...\n"
    "    ]\n"
    "  }\n"
    "}\n"
    "\n"
    "LABEL SCALE (return an integer for each passage):\n"
    "0 — לא קשור (אין מענה לשאילתה)\n"
    "1 — קשר חלש (קשור נושאי בלבד; אין תשובה שימושית)\n"
    "2 — חלקי (יש תשובה חלקית/רמז משמעותי אך חסר רכיב מרכזי)\n"
    "3 — חזק (מכסה כמעט הכול; חסר פרט קטן)\n"
    "4 — מלא (מענה ברור, מפורש ושלם לשאילתה)\n"
    "\n"
    "GENERAL PRINCIPLES:\n"
    "- Judge ONLY by the text in each passage relative to the target query. Ignore plabel/pid, prior knowledge, or calibration labels.\n"
    "- Prefer precision over recall: if unsure between two adjacent labels, choose the LOWER one.\n"
    "- Empty or near-empty passages → 0.\n"
    "\n"
    "INTENT-AWARE RULES (determine the query intent first):\n"
    "A) FACTOID (מי/מה/מתי/איזה/כמה/באיזו שנה/שם המשפחה/מה צבע/הרומן הראשון):\n"
    "   • Give 4 ONLY if the passage EXPLICITLY states the exact answer string and the relation.\n"
    "     Examples: year appears as the founding/opening year; the exact color word; the exact family name; the title explicitly presented as 'the first'.\n"
    "   • If the answer string appears but the specific relation ('first', 'year of', etc.) is missing/ambiguous → 2.\n"
    "   • Topical bios, link lists, metadata, film adaptations, or general mentions WITHOUT the answer string → 0–1 (usually 0).\n"
    "   • 'First/ראשון' requires explicit lexical evidence (e.g., 'הראשון', 'first', 'נוסד בשנת …' for date). Do NOT infer from context alone.\n"
    "\n"
    "B) MEMBERSHIP / SET (e.g., אילו יצירות/איפה לקרוא/אילו עובדו לקולנוע):\n"
    "   • A passage that clearly lists or states the requested membership/property with the queried item can be 3–4.\n"
    "   • Link hubs (e.g., Project Gutenberg/Ben-Yehuda) can be 3–4 ONLY if the query is about where to read/access; otherwise 0–1.\n"
    "\n"
    "C) EXPLANATORY (WHY/HOW/סיבות/כיצד):\n"
    "   • 4 requires clear, specific reasons/steps that answer the query; vague or partial mentions → 2–3.\n"
    "\n"
    "SPECIAL CASES:\n"
    "- Film/TV adaptations are relevant (up to 3–4) ONLY if the query is about adaptations. For factoids about books/authors/dates, they are typically 0–1 unless they explicitly contain the answer string.\n"
    "- Dates/numbers/colors/names must appear explicitly in the passage for 3–4. If missing, cap at 2.\n"
    "- Duplicates/near-duplicates should receive consistent labels.\n"
    "\n"
    "OUTPUT (STRICT JSON; no markdown/comments/no extra keys):\n"
    "{ \"labels\": [ <0|1|2|3|4>, <0|1|2|3|4>, ... ] }\n"
    "The number of labels MUST equal target.to_label length, in the same order.\n"
)

# ---------------- IO helpers ----------------
def iter_jsonl(p: str):
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s: yield json.loads(s)

def load_corpus_map(path: str) -> Dict[str, str]:
    mp = {}
    if not Path(path).exists():
        return mp
    _dbg(f"[IO] Loading corpus map from {path} …")
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s: continue
            o = json.loads(s)
            uid = o.get("uuid") or o.get("id")
            if uid:
                mp[uid] = (o.get("passage") or o.get("text") or "").strip()
                n += 1
                if DEBUG and n % 50000 == 0:
                    _dbg(f"[IO]  … {n:,} passages")
    _dbg(f"[IO] Loaded {n:,} corpus passages")
    return mp

def load_package_map(path: str) -> Dict[str, Dict[str, Any]]:
    m = {}
    for o in iter_jsonl(path):
        qid = o.get("query_uuid")
        if qid: m[qid] = o
    return m

def load_pseudo_map(path: str) -> Dict[str, Dict[str, Any]]:
    """query_uuid -> {'query': str, 'labels': [ {'paragraph_uuid','label','rank','score'} ... ] }"""
    m = {}
    for o in iter_jsonl(path):
        qid = o.get("query_uuid")
        if not qid: continue
        m[qid] = {"query": (o.get("query") or "").strip(), "labels": o.get("labels") or []}
    return m

def pick_diverse_by_label(examples: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    by = {i: [] for i in range(5)}
    for ex in examples or []:
        try: L = int(ex.get("label", -1))
        except: L = -1
        if 0 <= L <= 4:
            by[L].append(ex)
    out = []
    for L in [4,3,2,1,0]:
        if len(out) >= k: break
        if by[L]:
            out.append(by[L][0])
    return out

# Build passage lookup from package.pipeline_topk (useful if PSEUDO lacks text)
def build_passage_from_package(package_map: Dict[str, Dict[str, Any]]) -> Dict[str, str]:
    pm = {}
    for row in package_map.values():
        for c in (row.get("pipeline_topk") or []):
            pid = c.get("paragraph_uuid")
            ptx = (c.get("passage") or "").strip()
            if pid and ptx:
                pm[pid] = ptx
    return pm

# ---------------- Payload & model ----------------
def make_payload_for_query(
    qid: str,
    pseudo_row: Dict[str, Any],
    package_map: Dict[str, Dict[str, Any]],
    corpus_map: Dict[str, str],
    package_pass_map: Dict[str, str],
    max_sims: int,
    max_examples_per_sim: int
) -> Dict[str, Any]:
    pkg = package_map.get(qid, {})
    qtext = pseudo_row.get("query") or pkg.get("query") or ""

    # calibration blocks (group by similar query)
    calibration = []
    sims = pkg.get("supporting_positive_queries") or []
    if sims:
        n = len(sims) if max_sims <= 0 else min(max_sims, len(sims))
        for sim in sims[:n]:
            sq = (sim.get("similar_query") or "").strip()
            exs = pick_diverse_by_label(sim.get("diverse_labeled_examples") or [], k=max_examples_per_sim)
            calibration.append({
                "similar_query": sq,
                "examples": [{"label": int(e.get("label",0)), "passage": (e.get("passage") or "").strip()}
                             for e in exs]
            })

    # target to_label: ALL pseudo-labeled passages (keep order)
    to_label = []
    for it in pseudo_row.get("labels") or []:
        pid = it.get("paragraph_uuid")
        plabel = int(it.get("label", 0))
        passage = corpus_map.get(pid, "") or package_pass_map.get(pid, "")
        to_label.append({"passage": passage, "pid": pid, "plabel": plabel})

    return {"calibration": calibration, "target": {"query": qtext, "to_label": to_label}}

def parse_json_loose(text: str) -> Dict[str, Any]:
    try: return json.loads(text)
    except Exception: pass
    t = text.replace("```json", "").replace("```", "").strip()
    try: return json.loads(t)
    except Exception: pass
    a, b = t.find("{"), t.rfind("}")
    if a >= 0 and b > a:
        return json.loads(t[a:b+1])
    raise ValueError("Could not parse JSON from model output")

async def call_model(client: AsyncOpenAI, model: str, payload: Dict[str, Any]) -> List[int]:
    backoff = 1.0
    last_err = None
    for attempt in range(MAX_RETRY):
        try:
            _dbg(f"[CALL] attempt {attempt+1} | cal={len(payload.get('calibration', []))} | to_label={len(payload.get('target',{}).get('to_label', []))}")
            resp = await client.responses.create(
                model=model,
                input=[
                    {"role": "developer", "content": [{"type": "input_text", "text": DEV_PROMPT}]},
                    {"role": "user",      "content": [{"type": "input_text", "text": json.dumps(payload, ensure_ascii=False)}]},
                ],
                text={"format": {"type": "text"}, "verbosity": "medium"},
                reasoning={"effort": "medium", "summary": "auto"},
                tools=[],
                store=False
            )
            text = getattr(resp, "output_text", None)
            if not text and hasattr(resp, "output") and isinstance(resp.output, list):
                parts = []
                for item in resp.output:
                    for c in getattr(item, "content", []) or []:
                        if isinstance(getattr(c, "text", None), str):
                            parts.append(c.text)
                if parts: text = "\n".join(parts)
            if not text and hasattr(resp, "content") and isinstance(resp.content, list):
                parts = []
                for c in resp.content:
                    if isinstance(getattr(c, "text", None), str):
                        parts.append(c.text)
                if parts: text = "\n".join(parts)
            if not text:
                raise RuntimeError("Empty response")
            _dbg(f"[CALL] raw (first 300): {text[:300].replace(os.linesep,' ')}{'…' if len(text)>300 else ''}")
            out = parse_json_loose(text)
            labels = out.get("labels")
            if not isinstance(labels, list) or not all(isinstance(x, int) for x in labels):
                raise ValueError("Invalid labels format")
            for x in labels:
                if x < 0 or x > 4:
                    raise ValueError(f"Label out of range: {x}")
            return labels
        except Exception as e:
            _dbg(f"[ERR] model call error attempt {attempt+1}: {e}")
            last_err = e
            if attempt == MAX_RETRY - 1:
                raise
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 8.0)
    raise last_err or RuntimeError("Unknown error")

# ---------------- Main ----------------
async def main():
    assert Path(PSEUDO_JSONL).exists(),      f"Missing {PSEUDO_JSONL}"
    assert Path(LLM_PACKAGE_JSONL).exists(), f"Missing {LLM_PACKAGE_JSONL}"
    if not Path(CORPUS_JSONL).exists():
        print(f"[WARN] {CORPUS_JSONL} not found; fallback passages may be missing.", flush=True)

    pseudo_map  = load_pseudo_map(PSEUDO_JSONL)
    package_map = load_package_map(LLM_PACKAGE_JSONL)
    corpus_map  = load_corpus_map(CORPUS_JSONL)
    package_pass_map = build_passage_from_package(package_map)

    client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    Path(OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)

    done = set()
    if Path(OUT_JSONL).exists():
        _dbg(f"[IO] Resuming from existing OUT_JSONL: {OUT_JSONL}")
        for row in iter_jsonl(OUT_JSONL):
            qid = row.get("query_uuid")
            if qid: done.add(qid)
        _dbg(f"[IO] already have {len(done)} queries")

    out_fh = open(OUT_JSONL, "a" if done else "w", encoding="utf-8")
    sem = asyncio.Semaphore(CONCURRENCY)
    lock = asyncio.Lock()

    async def worker(qid: str, prow: Dict[str, Any]):
        if qid in done:
            _dbg(f"[DBG] skip done qid={qid}")
            return
        payload = make_payload_for_query(
            qid, prow, package_map, corpus_map, package_pass_map,
            max_sims=MAX_SIMS, max_examples_per_sim=MAX_EXAMPLES_PER_SIM
        )
        to_label = payload.get("target", {}).get("to_label", [])
        if not to_label:
            _dbg(f"[DBG] no passages to label for qid={qid}")
            return
        labels = await call_model(client, MODEL, payload)
        if len(labels) != len(to_label):
            _dbg(f"[ERR] length mismatch for qid={qid}: got {len(labels)} vs {len(to_label)}")
            return
        out_labels = []
        for lab, item in zip(labels, to_label):
            out_labels.append({
                "paragraph_uuid": item.get("pid"),
                "label": int(lab),
                "old_label": int(item.get("plabel", 0))
            })
        row = {"query_uuid": qid, "query": (prow.get("query") or ""), "labels": out_labels}
        async with lock:
            out_fh.write(json.dumps(row, ensure_ascii=False) + "\n")
            out_fh.flush()

    tasks = []
    for qid, prow in pseudo_map.items():  # exactly the 105 queries
        async def run(qid=qid, prow=prow):
            async with sem:
                try:
                    await worker(qid, prow)
                except Exception as e:
                    print(f"[ERR] qid={qid}: {e}", file=sys.stderr)
        tasks.append(asyncio.create_task(run()))

    print(f"[IO] Scheduling {len(tasks)} queries…")
    await asyncio.gather(*tasks)
    out_fh.close()

    print(f"[DONE] wrote → {OUT_JSONL}")
    if PRINT_FIRST > 0 and Path(OUT_JSONL).exists():
        print("\n[PREVIEW]")
        for i, row in enumerate(iter_jsonl(OUT_JSONL)):
            if i >= PRINT_FIRST: break
            s = json.dumps(row, ensure_ascii=False)
            print(s[:1000] + ("…" if len(s) > 1000 else ""))

if __name__ == "__main__":
    asyncio.run(main())


## Step 8: Merge Labels into Training Data

Augments `hsrc_train.jsonl` with the LLM-assigned labels, creating `hsrc_train_augmented.jsonl`.

In [ ]:
# augment_hsrc_train_with_llm.py
# Merge LLM-labeled results into the original hsrc_train.jsonl and write a new augmented file.
#
# Inputs (env-overridable):
#   TRAIN_JSONL=/content/hsrc_train.jsonl
#   CORPUS_JSONL=/content/hsrc_corpus.jsonl
#   PSEUDO_JSONL=./zero_pos_pseudo_labels.jsonl
#
# Output:
#   OUT_JSONL=/content/hsrc_train_augmented.jsonl
#
# Options:
#   APPEND_ONLY_POSITIVES=0   # set 1 to append only label>0 new passages
#   DRY_RUN=0                 # set 1 to print stats only (no file write)
#   DEBUG=1                   # verbose prints

import os, json, sys
from pathlib import Path
from typing import Dict, Any, Iterable, List, Tuple

TRAIN_JSONL   = os.getenv("TRAIN_JSONL",   "./hsrc_train.jsonl")
CORPUS_JSONL  = os.getenv("CORPUS_JSONL",  "./hsrc_corpus.jsonl")
PSEUDO_JSONL  = os.getenv("PSEUDO_JSONL",  "./zero_pos_pseudo_labels.pass2.jsonl")
OUT_JSONL     = os.getenv("OUT_JSONL",     "./hsrc_train_augmented.jsonl")

APPEND_ONLY_POSITIVES = bool(int(os.getenv("APPEND_ONLY_POSITIVES", "0")))
DRY_RUN               = bool(int(os.getenv("DRY_RUN", "0")))
DEBUG                 = bool(int(os.getenv("DEBUG", "1")))

def dbg(*args):
    if DEBUG:
        print(*args, flush=True)

def read_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                yield json.loads(s)

def write_jsonl(path: str, rows: Iterable[Dict[str, Any]]):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def load_corpus_map(path: str) -> Dict[str, str]:
    dbg(f"[IO] Loading corpus from {path} …")
    mp = {}
    n = 0
    for o in read_jsonl(path):
        uid = o.get("uuid") or o.get("id")
        if uid:
            mp[uid] = (o.get("passage") or o.get("text") or "").strip()
            n += 1
            if DEBUG and n % 50000 == 0:
                dbg(f"[IO]  … {n:,} passages")
    dbg(f"[IO] Loaded {n:,} passages.")
    return mp

def load_pseudo_map(path: str) -> Dict[str, List[Dict[str, Any]]]:
    """query_uuid -> list of {'paragraph_uuid','label', 'rank', 'score'} (order preserved)"""
    dbg(f"[IO] Loading pseudo labels from {path} …")
    mp = {}
    n_queries, n_labels = 0, 0
    for row in read_jsonl(path):
        qid = row.get("query_uuid")
        labs = row.get("labels") or []
        if qid:
            mp[qid] = []
            for it in labs:
                pid = it.get("paragraph_uuid")
                lab = it.get("label")
                if pid is None or lab is None:
                    continue
                try:
                    lab = int(lab)
                except Exception:
                    continue
                mp[qid].append({"paragraph_uuid": pid, "label": lab, "rank": it.get("rank"), "score": it.get("score")})
                n_labels += 1
            n_queries += 1
    dbg(f"[IO] Pseudo map: {n_queries} queries, {n_labels} total labels.")
    return mp

def next_free_index(paragraphs: Dict[str, Any], labels: Dict[str, Any]) -> int:
    """Assumes contiguous indices from 0; returns next index after the last contiguous pair present in both dicts."""
    i = 0
    while True:
        pk, lk = f"paragraph_{i}", f"target_action_{i}"
        if pk in paragraphs and lk in labels:
            i += 1
            continue
        break
    return i

# === NEW ===
def _parse_label(v) -> int:
    try:
        if isinstance(v, int):
            return v
        if isinstance(v, str) and v.strip() != "":
            # allow "0", "1", "2.0"
            return int(v) if v.isdigit() else int(float(v))
    except Exception:
        pass
    return 0

def merge_one_query(
    o: Dict[str, Any],
    pseudo_for_q: List[Dict[str, Any]],
    corpus_map: Dict[str, str],
    append_only_pos: bool
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Update a single train item with pseudo labels.
    Returns (updated_object, stats_dict)

    stats keys:
      - updated_existing, appended_new, skipped_no_passage, skipped_by_flag
      - existing_zero_to_pos_total, existing_zero_to_pos_by_label {1..4}
      - unlabeled_pos_total, unlabeled_pos_by_label {1..4}   # pseudo for new (not in original), regardless of append
    """
    paragraphs = o.get("paragraphs") or {}
    labels     = o.get("target_actions") or {}

    # Build reverse map: paragraph_uuid -> index
    uuid_to_idx = {}
    for i in range(1000):
        pk, lk = f"paragraph_{i}", f"target_action_{i}"
        if pk not in paragraphs or lk not in labels:
            break
        pid = (paragraphs[pk].get("uuid") or paragraphs[pk].get("id"))
        if pid:
            uuid_to_idx[pid] = i

    # Find next contiguous index to append
    i_next = next_free_index(paragraphs, labels)

    # base stats
    stats = {
        'updated_existing': 0,
        'appended_new': 0,
        'skipped_no_passage': 0,
        'skipped_by_flag': 0,
        # NEW: detailed counters
        'existing_zero_to_pos_total': 0,
        'existing_zero_to_pos_by_label': {1:0, 2:0, 3:0, 4:0},
        'unlabeled_pos_total': 0,
        'unlabeled_pos_by_label': {1:0, 2:0, 3:0, 4:0},
    }

    # Merge labels (preserve pseudo order)
    for it in pseudo_for_q:
        pid = it["paragraph_uuid"]
        lab = int(it["label"])

        # optionally skip appending negatives, but still allow updating existing ones
        if append_only_pos and lab <= 0 and pid not in uuid_to_idx:
            stats['skipped_by_flag'] += 1
            continue

        if pid in uuid_to_idx:
            idx = uuid_to_idx[pid]
            lk = f"target_action_{idx}"
            old_lab = _parse_label(labels.get(lk, 0))  # === NEW === get previous human label
            # record zero->positive transitions for EXISTING items
            if old_lab == 0 and lab > 0:
                stats['existing_zero_to_pos_total'] += 1
                if lab in (1,2,3,4):
                    stats['existing_zero_to_pos_by_label'][lab] += 1
            # perform update
            labels[lk] = int(lab)
            stats['updated_existing'] += 1
        else:
            # === NEW === count unlabeled (not in original) positives regardless of append success
            if lab > 0:
                stats['unlabeled_pos_total'] += 1
                if lab in (1,2,3,4):
                    stats['unlabeled_pos_by_label'][lab] += 1

            # append new candidate; need passage text
            passage = corpus_map.get(pid, "")
            if passage == "":
                stats['skipped_no_passage'] += 1
                continue
            pk, lk = f"paragraph_{i_next}", f"target_action_{i_next}"
            paragraphs[pk] = {"uuid": pid, "passage": passage}
            labels[lk] = int(lab)
            uuid_to_idx[pid] = i_next
            i_next += 1
            stats['appended_new'] += 1

    # write back
    o["paragraphs"] = paragraphs
    o["target_actions"] = labels
    return o, stats

def main():
    # checks
    assert Path(TRAIN_JSONL).exists(),  f"Missing TRAIN_JSONL: {TRAIN_JSONL}"
    assert Path(PSEUDO_JSONL).exists(), f"Missing PSEUDO_JSONL: {PSEUDO_JSONL}"
    if not Path(CORPUS_JSONL).exists():
        print(f"[WARN] CORPUS_JSONL not found at {CORPUS_JSONL}. Appending new passages without text will be skipped.", file=sys.stderr)

    # load maps
    pseudo_map  = load_pseudo_map(PSEUDO_JSONL)
    corpus_map  = load_corpus_map(CORPUS_JSONL) if Path(CORPUS_JSONL).exists() else {}

    # merge
    out_rows = []
    n_total, n_hit = 0, 0
    totals = {
        'updated_existing':0,
        'appended_new':0,
        'skipped_no_passage':0,
        'skipped_by_flag':0,
        # NEW aggregated counters:
        'existing_zero_to_pos_total': 0,
        'existing_zero_to_pos_by_label': {1:0, 2:0, 3:0, 4:0},
        'unlabeled_pos_total': 0,
        'unlabeled_pos_by_label': {1:0, 2:0, 3:0, 4:0},
    }

    dbg("[RUN] Merging labels into training set …")
    for o in read_jsonl(TRAIN_JSONL):
        n_total += 1
        qid = o.get("query_uuid") or o.get("id")
        if qid in pseudo_map:
            n_hit += 1
            new_o, st = merge_one_query(o, pseudo_map[qid], corpus_map, APPEND_ONLY_POSITIVES)
            # accumulate base stats
            for k in ['updated_existing','appended_new','skipped_no_passage','skipped_by_flag',
                      'existing_zero_to_pos_total','unlabeled_pos_total']:
                totals[k] += st[k]
            # accumulate dict counters
            for L in (1,2,3,4):
                totals['existing_zero_to_pos_by_label'][L] += st['existing_zero_to_pos_by_label'][L]
                totals['unlabeled_pos_by_label'][L]      += st['unlabeled_pos_by_label'][L]
            out_rows.append(new_o)
        else:
            out_rows.append(o)

        if DEBUG and n_total % 200 == 0:
            dbg(f"[RUN]  … processed {n_total:,} items (matched {n_hit:,})")

    # stats
    print("\n[SUMMARY]")
    print(f" Train items processed:   {n_total:,}")
    print(f" Train items updated:     {n_hit:,}")
    print(f"   - updated_existing:    {totals['updated_existing']:,}")
    print(f"   - appended_new:        {totals['appended_new']:,}")
    print(f"   - skipped_no_passage:  {totals['skipped_no_passage']:,}")
    print(f"   - skipped_by_flag:     {totals['skipped_by_flag']:,}")
    print(f" Append-only-positives:   {APPEND_ONLY_POSITIVES}")

    # === NEW: quality-impact counters ===
    print("\n[QUALITY COUNTS]")
    # Existing items that were 0 -> positive
    ezp = totals['existing_zero_to_pos_total']
    ezp_by = totals['existing_zero_to_pos_by_label']
    print(f" Existing passages originally labeled 0 → now positive: {ezp:,}")
    print(f"   by new label: 1→{ezp_by[1]:,} | 2→{ezp_by[2]:,} | 3→{ezp_by[3]:,} | 4→{ezp_by[4]:,}")

    # Unlabeled (not in original) pseudo-labeled positives (regardless of append success)
    ulp = totals['unlabeled_pos_total']
    ulp_by = totals['unlabeled_pos_by_label']
    print(f" Unlabeled (new) passages pseudo-labeled positive: {ulp:,}")
    print(f"   by new label: 1→{ulp_by[1]:,} | 2→{ulp_by[2]:,} | 3→{ulp_by[3]:,} | 4→{ulp_by[4]:,}")

    if DRY_RUN:
        print("\n[DRY_RUN] Not writing output file (set DRY_RUN=0 to write).")
        return

    # write
    write_jsonl(OUT_JSONL, out_rows)
    print(f"\n[DONE] Wrote augmented training set → {OUT_JSONL}")

if __name__ == "__main__":
    main()


## Step 9: Diff Labels (Optional)

Prints side-by-side comparison of original vs LLM-assigned labels for verification.

In [ ]:
# diff_labels_print.py
# Print, for each query, the passages whose label changed: old_label -> new_label,
# with full passage text. Optionally filter to ONLY changes where new_label ∈ {…}.

import os, json, re
from pathlib import Path
from typing import Dict, Any, Iterable, Tuple

TRAIN_ORIG     = os.getenv("TRAIN_ORIG", "./hsrc_train.jsonl")
TRAIN_NEW      = os.getenv("TRAIN_NEW",  "./hsrc_train_augmented.jsonl")
CORPUS_JSONL   = os.getenv("CORPUS_JSONL", "./hsrc_corpus.jsonl")
ONLY_PRINT_CHANGED = bool(int(os.getenv("ONLY_PRINT_CHANGED", "1")))
N_LIMIT        = int(os.getenv("N_LIMIT", "0"))  # 0 = all

# NEW: comma/space-separated list of target new labels (e.g., "2,3,4").
# If empty => no filtering (show all changed).
_filter_env = os.getenv("FILTER_NEW_LABELS", "4").strip()
TARGET_NEW_LABELS = None
if _filter_env:
    TARGET_NEW_LABELS = {int(x) for x in re.split(r"[,\s]+", _filter_env) if x != ""}

def read_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                yield json.loads(s)

def parse_label(v) -> int:
    try:
        if isinstance(v, int):
            return v
        if isinstance(v, str) and v.isdigit():
            return int(v)
        return int(float(v))
    except Exception:
        return 0

def extract_label_maps(row: Dict[str, Any]) -> Tuple[Dict[str, int], Dict[str, int]]:
    """
    Returns:
      labels_map: paragraph_uuid -> label
      idx_map:    paragraph_uuid -> index (paragraph_i / target_action_i)
    """
    labels_map, idx_map = {}, {}
    paras = row.get("paragraphs", {}) or {}
    labs  = row.get("target_actions", {}) or {}
    for i in range(1000):
        pk, lk = f"paragraph_{i}", f"target_action_{i}"
        if pk not in paras or lk not in labs:
            break
        pid = paras[pk].get("uuid") or paras[pk].get("id")
        if not pid:
            continue
        labels_map[pid] = parse_label(labs[lk])
        idx_map[pid] = i
    return labels_map, idx_map

def load_corpus_map(path: str) -> Dict[str, str]:
    mp = {}
    if not Path(path).exists():
        return mp
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            o = json.loads(s)
            uid = o.get("uuid") or o.get("id")
            if uid:
                mp[uid] = (o.get("passage") or o.get("text") or "").strip()
    return mp

def main():
    assert Path(TRAIN_ORIG).exists(), f"Not found: {TRAIN_ORIG}"
    assert Path(TRAIN_NEW).exists(),  f"Not found: {TRAIN_NEW}"
    if not Path(CORPUS_JSONL).exists():
        print(f"[WARN] corpus file not found at {CORPUS_JSONL}; passages may be missing.", flush=True)

    # Load files keyed by query_uuid
    orig_by_qid, new_by_qid = {}, {}
    for row in read_jsonl(TRAIN_ORIG):
        qid = row.get("query_uuid") or row.get("id")
        if qid: orig_by_qid[qid] = row
    for row in read_jsonl(TRAIN_NEW):
        qid = row.get("query_uuid") or row.get("id")
        if qid: new_by_qid[qid] = row

    corpus_map = load_corpus_map(CORPUS_JSONL)

    printed_queries = 0
    total_changed = 0
    total_unchanged_overlap = 0

    for qid, new_row in new_by_qid.items():
        if qid not in orig_by_qid:
            continue
        orig_row = orig_by_qid[qid]

        orig_labels, orig_idx = extract_label_maps(orig_row)
        new_labels,  new_idx  = extract_label_maps(new_row)

        overlap_pids = sorted(set(orig_labels.keys()) & set(new_labels.keys()))

        changed_items = []
        unchanged_items = []
        for pid in overlap_pids:
            old_lab = orig_labels[pid]
            new_lab = new_labels[pid]
            # Only consider changes
            if old_lab != new_lab:
                # If filter is set, keep only those where new_lab ∈ TARGET_NEW_LABELS
                if TARGET_NEW_LABELS is not None and new_lab not in TARGET_NEW_LABELS:
                    continue
                passage = corpus_map.get(
                    pid,
                    new_row.get("paragraphs", {}).get(f"paragraph_{new_idx.get(pid, -1)}", {}).get("passage", "")
                )
                changed_items.append({
                    "paragraph_uuid": pid,
                    "old_label": old_lab,
                    "new_label": new_lab,
                    "old_idx": orig_idx.get(pid),
                    "new_idx": new_idx.get(pid),
                    "passage": passage
                })
            else:
                if not ONLY_PRINT_CHANGED:
                    passage = corpus_map.get(pid, "")
                    unchanged_items.append({
                        "paragraph_uuid": pid,
                        "label": old_lab,
                        "idx": new_idx.get(pid),
                        "passage": passage
                    })

        # If filtering left no changed items, skip printing this query
        if ONLY_PRINT_CHANGED and not changed_items:
            continue

        printed_queries += 1
        total_changed += len(changed_items)
        total_unchanged_overlap += len(unchanged_items)

        query_text = (new_row.get("query") or orig_row.get("query") or "").strip()

        print("=" * 120)
        print(f"query_uuid: {qid}")
        print(f"query: {query_text}\n")

        if changed_items:
            print(f"Changed passages (filtered): {len(changed_items)}"
                  + (f" | new_label ∈ {sorted(TARGET_NEW_LABELS)}" if TARGET_NEW_LABELS is not None else ""))
            for j, it in enumerate(changed_items, 1):
                print(f"  {j:02d}. paragraph_uuid={it['paragraph_uuid']}  |  {it['old_label']} -> {it['new_label']}  |  old_idx={it['old_idx']}  new_idx={it['new_idx']}")
                print("      passage (FULL):")
                print(it["passage"] if it["passage"] else "(passage text not found)")
                print("-" * 120)
        else:
            print("Changed passages (filtered): 0")

        if not ONLY_PRINT_CHANGED and unchanged_items:
            print("\nUnchanged overlapping passages:")
            for j, it in enumerate(unchanged_items, 1):
                print(f"  {j:02d}. paragraph_uuid={it['paragraph_uuid']}  |  label={it['label']}  |  idx={it['idx']}")
                print("      passage (FULL):")
                print(it["passage"] if it["passage"] else "(passage text not found)")
                print("-" * 120)

        if N_LIMIT and printed_queries >= N_LIMIT:
            break

    print("=" * 120)
    print(
        f"Done. Queries printed: {printed_queries} | Changed passages total (filtered): {total_changed}"
        + ("" if ONLY_PRINT_CHANGED else f" | Unchanged-overlap: {total_unchanged_overlap}")
    )

if __name__ == "__main__":
    main()
